In [1]:
# 从这里开始我们思考如何实现loss函数部分

In [2]:
import torch 

from pytorch3d.io import load_objs_as_meshes
from pytorch3d.structures import Meshes
import drone_renderer

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

render = drone_renderer.DroneRenderer(mesh_path= "data/sample/sample.obj",device= device)
mesh = render.mesh

B = 4  # 批量大小
P = torch.rand(B, 3,device=device) *10.0  # 假设的无人机中心点坐标



In [3]:
from pytorch3d.ops import sample_points_from_meshes, knn_points

# 1. 将 Mesh 转换为点云 (Point Cloud) 以供 KNN 使用
# 我们直接使用 render 对象中已经加载好的 mesh
# num_samples 决定了障碍物点云的密度，点越多计算越精确但开销越大
num_samples = 20000 
obstacle_pcd = sample_points_from_meshes(render.mesh, num_samples=num_samples)

print(f"生成的障碍物点云形状: {obstacle_pcd.shape}")  # 预期: (1, 20000, 3)

生成的障碍物点云形状: torch.Size([1, 20000, 3])


In [4]:
p1 = P.unsqueeze(1)  # 形状变为 (B, 1, 3)
p2 = obstacle_pcd  # 形状为 (1, N, 3)，N 是点云中的点数
p2 = p2.expand(B,-1,-1)  # 扩展为 (B, N, 3) 以匹配无人机批量大小
print(f"扩展后的障碍物点云形状: {p2.shape}")  # 预期: (B, 20000, 3)

dists = knn_points(p1, p2, K=1)
print(f"计算得到的最近距离形状: {dists.dists.shape}")  # 预期: (B, 1, 1)
print("距离结果",dists)
print("纯距离",dists.dists)  # 打印距离值以检查
print("形状",dists.dists.shape)
dists.dists.squeeze(-1)
print("去掉最后一个维度后的形状",dists.dists.squeeze(-1).shape)



扩展后的障碍物点云形状: torch.Size([4, 20000, 3])
计算得到的最近距离形状: torch.Size([4, 1, 1])
距离结果 KNN(dists=tensor([[[ 0.1384]],

        [[ 0.2489]],

        [[ 2.0507]],

        [[15.2876]]], device='cuda:0'), idx=tensor([[[ 2747]],

        [[12072]],

        [[10815]],

        [[ 2926]]], device='cuda:0'), knn=None)
纯距离 tensor([[[ 0.1384]],

        [[ 0.2489]],

        [[ 2.0507]],

        [[15.2876]]], device='cuda:0')
形状 torch.Size([4, 1, 1])
去掉最后一个维度后的形状 torch.Size([4, 1])


In [5]:

def calc_min_distance(drone_pos, obstacle_pcd):
    """
    计算无人机中心点与障碍物点云之间的最短距离 (Single Step)
    arges:
        drone_pos: (B, 3) 无人机中心点坐标
        obstacle_pcd: (1 or B, N, 3) 障碍物点云
    returns:
        dists: (B,) 每个无人机到障碍物的最短距离
    """
    p1 = drone_pos.unsqueeze(1) 
    B = p1.shape[0]
    
    if obstacle_pcd.shape[0] != B:
        obstacle_pcd_expanded = obstacle_pcd.expand(B, -1, -1)
    else:
        obstacle_pcd_expanded = obstacle_pcd
        
    result = knn_points(p1, obstacle_pcd_expanded, K=1)
    sq_dists = result.dists.squeeze(-1) # (B, 1) -> (B,)
    dists = torch.sqrt(sq_dists + 1e-6).squeeze(-1) 
    return dists

def distance_to_obj(drone_pos, obstacle_pcd):
    """
    计算无人机到障碍物的最短距离
    Args:
        drone_pos: (B, 3) 无人机位置
        obstacle_pcd: (1, N, 3) 障碍物点云
    Returns:
        dists: (B,) 距离
    """
    return calc_min_distance(drone_pos, obstacle_pcd)

def vec_to_obj(drone_pos, obstacle_pcd):
    """
    计算无人机到最近障碍物点的向量
    Args:
        drone_pos: (B, 3) 无人机位置
        obstacle_pcd: (1, N, 3) 障碍物点云
    Returns:
        vecs: (B, 3) 向量，从无人机到最近点
    """
    p1 = drone_pos.unsqueeze(1)  # (B, 1, 3)
    B = p1.shape[0]
    
    if obstacle_pcd.shape[0] != B:
        obstacle_pcd_expanded = obstacle_pcd.expand(B, -1, -1)
    else:
        obstacle_pcd_expanded = obstacle_pcd
        
    result = knn_points(p1, obstacle_pcd_expanded, K=1)
    idx = result.idx.squeeze(-1).squeeze(-1)  # (B,)
    
    # 获取最近的点
    nearest_points = obstacle_pcd_expanded[torch.arange(B), idx]  # (B, 3)
    
    # 向量：最近点 - 无人机位置
    vecs = nearest_points - drone_pos
    return vecs

# 测试函数
min_distances = distance_to_obj(P, obstacle_pcd)
vecs = vec_to_obj(P, obstacle_pcd)
print("距离:", min_distances)
print("向量:", vecs)

距离: tensor([0.3720, 0.4989, 1.4320, 3.9099], device='cuda:0')
向量: tensor([[ 0.2710,  0.2493, -0.0528],
        [ 0.2227,  0.3132,  0.3181],
        [ 1.4022, -0.2694,  0.1086],
        [-0.2076, -3.9040, -0.0602]], device='cuda:0')


# 以上代码实现了计算无人机到障碍物点云的最短距离和最近点向量的功能，并进行了测试。

In [6]:
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [7]:
def barrier(x: torch.Tensor, v_to_pt):
    return (v_to_pt * (1 - x).relu().pow(2)).mean()

In [ ]:
from drone_env import DroneSimulator
from model import Model
B = 4  # 批量大小
dt = 0.02  # 时间步长
num_steps = 200  # 模拟步数

# 初始化仿真环境
# mesh_path: 从 ipynb 目录看，数据在 ../data
# 完全暴露参数的版本
env = DroneSimulator(
    batch_size=B, 
    dt=dt, 
    mesh_path="../data/sample/sample.obj",
    device=device ,
    enable_airmode=True,
    enable_induced_drag=False,
    noise_std=0.04,
    grad_decay=0.8,
    num_samples= 20000
)
model = Model(7,6).to(device)


# 随机生成一个恒定的期望指令用于测试
act = torch.rand(B, 3, device=device) * 15.0 
target_pos = torch.rand(B, 3, device=device) * 2.0  # 目标位置

for step in range(num_steps):
    rgb_images, depth_images = env.render(camera_pitch=10.0)
    target_vector = target_pos - env.p
    state = env.step(act_cmd=act, target_pos_vector=target_vector)
